# ACHTUNG: In diesem Notebook werden händisch Werte nachgetragen. Am besten Notebook nachvollziehen und mit dem nächsten Datensatz im nächsten Notebook weitermachen!

In [17]:
import pandas as pd
import os

In [3]:
df = pd.read_pickle('../../data/processed/12_cleaned_master_data.pkl')

In [24]:
df.isna().sum()

race                                  0
year                                  0
url                                   0
rank                                  0
rider_url                             0
time_gap                              0
birthdate                             0
height                              910
name                                  0
nationality                           0
weight                              910
url_name                              0
departure                             0
arrival                               0
distance                              0
vertical_meters                    3727
won_how                               0
avg_speed                             0
race_ranking                          0
one_day_races                         0
gc                                    0
time_trial                            0
sprint                                0
climber                               0
hills                                 0


### Vertical Meters NAs

- vertical meters und profile_score scheinen zusammen zu hängen
- da sich Score aus den Höhenmetern ergibt droppen wir die Spalte



In [5]:
df.drop(columns=['profile_score'], inplace=True, errors='ignore')

In [6]:
missing_stages = df[df['vertical_meters'].isna()]
unique_missing_stages = missing_stages[['year', 'race', 'stage_nr', 'url']].drop_duplicates()

print(f"Betroffene Fahrer-Zeilen gesamt: {len(missing_stages)}")
print(f"Anzahl betroffener Etappen gesamt: {len(unique_missing_stages)}")

Betroffene Fahrer-Zeilen gesamt: 5083
Anzahl betroffener Etappen gesamt: 35


Nur 35 Etappen sind betroffen 
- somit können wir händisch nachtragen was der Scraper nicht geschafft hat

In [7]:
unique_missing = missing_stages[['year', 'race', 'stage_nr', 'url']].drop_duplicates()
unique_missing = unique_missing.sort_values(by=['year', 'race', 'stage_nr'])

unique_missing

,year,race,stage_nr,url
78604,2005,giro-d-italia,20,https://www.procyclingstats.com/race/giro-d-it...
3462,2005,tour-de-france,21,https://www.procyclingstats.com/race/tour-de-f...
154246,2005,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
82186,2006,giro-d-italia,20,https://www.procyclingstats.com/race/giro-d-it...
157726,2006,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
85751,2007,giro-d-italia,21,https://www.procyclingstats.com/race/giro-d-it...
10125,2007,tour-de-france,20,https://www.procyclingstats.com/race/tour-de-f...
161297,2007,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
89358,2008,giro-d-italia,21,https://www.procyclingstats.com/race/giro-d-it...
13578,2008,tour-de-france,21,https://www.procyclingstats.com/race/tour-de-f...


Es fällt auf das alle Fehlenden Werte aus den Schlussetappen stammen.

Zuerst fällt auf, dass immernoch Zeitfahren dabei sind, welche wir nun entgültig eliminieren

In [8]:
#  Liste der 10  Zeitfahren als Bedingungen definieren
time_trial_blacklist = [
    {"race": "giro-d-italia", "year": 2008, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2009, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2010, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2011, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2012, "stage_nr": 21},
    {"race": "vuelta-a-espana", "year": 2014, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2017, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2020, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2021, "stage_nr": 21},
    {"race": "giro-d-italia", "year": 2022, "stage_nr": 21},
]

print("Zeilen vor dem Blacklist-Drop:", len(df))

# Jede Etappe aus der Blacklist nacheinander herausfiltern
for stage in time_trial_blacklist:
    df = df[
        ~(
            (df["race"] == stage["race"])
            & (df["year"] == stage["year"])
            & (df["stage_nr"] == stage["stage_nr"])
        )
    ].copy()

print("Zeilen nach dem Blacklist-Drop:", len(df))

Zeilen vor dem Blacklist-Drop: 197404
Zeilen nach dem Blacklist-Drop: 196048


In [9]:
missing_stages = df[df['vertical_meters'].isna()]
unique_missing = missing_stages[['year', 'race', 'stage_nr', 'url']].drop_duplicates()
unique_missing = unique_missing.sort_values(by=['year', 'race', 'stage_nr'])

unique_missing


,year,race,stage_nr,url
78604,2005,giro-d-italia,20,https://www.procyclingstats.com/race/giro-d-it...
3462,2005,tour-de-france,21,https://www.procyclingstats.com/race/tour-de-f...
154246,2005,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
82186,2006,giro-d-italia,20,https://www.procyclingstats.com/race/giro-d-it...
157726,2006,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
85751,2007,giro-d-italia,21,https://www.procyclingstats.com/race/giro-d-it...
10125,2007,tour-de-france,20,https://www.procyclingstats.com/race/tour-de-f...
161297,2007,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...
13578,2008,tour-de-france,21,https://www.procyclingstats.com/race/tour-de-f...
164579,2008,vuelta-a-espana,21,https://www.procyclingstats.com/race/vuelta-a-...


## Distance 0 values
- bevor wir nun die fehlenden Höhenmeterwerte lösen schauen wir uns die Verteilung der 0-values bei der Distanz der Etappen an


In [ ]:
missing_distance = df[(df['distance'] <= 0) | (df['distance'].isna())]

unique_missing_dist = missing_distance[['distance', 'race', 'stage_nr', 'url']].drop_duplicates()

if unique_missing_dist.empty:
    print("keine 0 oder na Werte gefunden")
else:
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)

    print(unique_missing_dist.to_string(index=False))

    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_colwidth')

    print("\n", len(unique_missing_dist))

 distance            race  stage_nr                                                                url
      0.0  tour-de-france        21  https://www.procyclingstats.com/race/tour-de-france/2005/stage-21
      0.0  tour-de-france        20  https://www.procyclingstats.com/race/tour-de-france/2006/stage-20
      0.0  tour-de-france        20  https://www.procyclingstats.com/race/tour-de-france/2007/stage-20
      0.0  tour-de-france        21  https://www.procyclingstats.com/race/tour-de-france/2008/stage-21
      0.0  tour-de-france        21  https://www.procyclingstats.com/race/tour-de-france/2009/stage-21
      0.0  tour-de-france        20  https://www.procyclingstats.com/race/tour-de-france/2010/stage-20
      0.0   giro-d-italia        20   https://www.procyclingstats.com/race/giro-d-italia/2005/stage-20
      0.0   giro-d-italia        20   https://www.procyclingstats.com/race/giro-d-italia/2006/stage-20
      0.0   giro-d-italia        21   https://www.procyclingstats.com/rac

- es handelt sich wieder um finale Etappen (27), die wir nun händisch noch hinzuschreiben
- Grund: Feld auf Website war 0 gesetzt, jedoch steht in der Überschrift noch einmal die Distanz

In [18]:
# Speichern einr csv zum händischen Nachtragen

unique_missing_dist['distance_recherchiert'] = ""

unique_missing_dist = unique_missing_dist.sort_values(by=['race', 'url'])

target_dir = "../../data/interim"
os.makedirs(target_dir, exist_ok=True)

target_path = os.path.join(target_dir, "27_etappen_distanz_recherche.csv")
unique_missing_dist.to_csv(target_path, index=False, sep=';')

In [23]:

#Mergen der eingefügten Werte

missing_complete = pd.read_csv('../../data/interim/27_etappen_distanz_recherche.csv', sep=';')


missing_complete['distance_recherchiert'] = pd.to_numeric(missing_complete['distance_recherchiert'], errors='coerce')

# neue Info über die URL an das Haupt-DataFrame mappen
df = df.merge(missing_complete[['url', 'distance_recherchiert']], on='url', how='left')


mask_zero = df['distance'] == 0
df.loc[mask_zero, 'distance'] = df.loc[mask_zero, 'distance_recherchiert'].combine_first(df.loc[mask_zero, 'distance'])

# Falls noch echte NaNs in distance waren, diese ebenfalls auffüllen
df['distance'] = df['distance'].combine_first(df['distance_recherchiert'])

# Die temporäre Spalte wieder löschen
df.drop(columns=['distance_recherchiert'], inplace=True)


print(f"Verbleibende Distanzen == 0:  {len(df[df['distance'] == 0])}")
print(f"Verbleibende Distanzen (NaN): {df['distance'].isna().sum()}")

Verbleibende Distanzen == 0:  0
Verbleibende Distanzen (NaN): 0


In [ ]:

etappe_21 = df[df["stage_nr"] == 21]

# Gruppieren und aggregieren: Zeilen-Count, Unique-Etappen-Count und Median
statistik_21_final = (
    etappe_21.groupby("race")
    .agg(
        beobachtungen_gesamt=(
            "vertical_meters",
            "count",
        ),
        anzahl_einzigartige_etappen=(
            "year",
            "nunique",
        ),
        median_hoehenmeter=(
            "vertical_meters",
            "median",
        ),
    )
    .reset_index()
)


statistik_21_final.columns = [
    "Rundfahrt (race)",
    "Fahrer-Zeilen (Observations)",
    "Einzigartige Etappen",
    "Median Höhenmeter",
]

print(statistik_21_final.to_string(index=False))

Rundfahrt (race)  Fahrer-Zeilen (Observations)  Einzigartige Etappen  Median Höhenmeter
   giro-d-italia                           416                     9              692.0
  tour-de-france                          2031                    16              732.0
 vuelta-a-espana                           589                    16              726.0


Imputation Median für Missing oder 0 -value Werter in vertical_meters 

In [ ]:
medians_21 = {
    'giro-d-italia': 692,
    'tour-de-france': 732,
    'vuelta-a-espana': 726
}

def impute_stage_21(row):
    if row['stage_nr'] == 21 and (pd.isna(row['vertical_meters']) or row['vertical_meters'] == 0):
        return medians_21.get(row['race'])

    return row['vertical_meters']


df['vertical_meters'] = df.apply(impute_stage_21, axis=1)


df['vertical_meters'] = df['vertical_meters'].astype('Int64')


etappe_21_check = df[df['stage_nr'] == 21]
print(f"Verbleibende NaNs auf Etappe 21: {etappe_21_check['vertical_meters'].isna().sum()}")
print(f"Verbleibende 0er auf Etappe 21:  {len(etappe_21_check[etappe_21_check['vertical_meters'] == 0])}")

Verbleibende NaNs auf Etappe 21: 0
Verbleibende 0er auf Etappe 21:  0


Prüfung anderer Etappen ohne vertical Meters

In [36]:
df['vertical_meters'].isna().sum()

np.int64(599)

In [ ]:

problem_elevation = df[
    ((df['vertical_meters'].isna()) | (df['vertical_meters'] == 0)) &
    (df['stage_nr'] != 21)
]


unique_problems = problem_elevation[['race', 'year', 'stage_nr', 'vertical_meters']].drop_duplicates()


if unique_problems.empty:
    print("Keine Etappen mit 0 oder NaNs gefunden.")
else:

    problem_statistik = unique_problems.groupby('stage_nr').size().reset_index(name='Anzahl betroffener Etappen')
    problem_statistik.columns = ['Etappen-Nummer (stage_nr)', 'Anzahl defekter Etappen']

    print(problem_statistik.to_string(index=False))

    print(f"\n {len(unique_problems)} Etappen fehlen noch")

 Etappen-Nummer (stage_nr)  Anzahl defekter Etappen
                        20                        5

 5 Etappen fehlen noch


In [39]:

etappe_20 = df[df["stage_nr"] == 20]


etappe_20_valid = etappe_20[(etappe_20["vertical_meters"].notna()) & (etappe_20["vertical_meters"] > 0)]


statistik_20_final = (
    etappe_20_valid.groupby("race")
    .agg(
        beobachtungen_gesamt=("vertical_meters", "count"),
        anzahl_einzigartige_etappen=("year", "nunique"),
        median_hoehenmeter=("vertical_meters", "median"),
    )
    .reset_index()
)

statistik_20_final.columns = [
    "Rundfahrt (race)",
    "Fahrer-Zeilen (Observations)",
    "Einzigartige Etappen",
    "Median Höhenmeter",
]

print(statistik_20_final.to_string(index=False))

Rundfahrt (race)  Fahrer-Zeilen (Observations)  Einzigartige Etappen  Median Höhenmeter
   giro-d-italia                          2572                    17             4254.0
  tour-de-france                          1416                     9             3471.0
 vuelta-a-espana                          2275                    15             4189.0


In [ ]:
missing_20 = df[
    ((df['vertical_meters'].isna()) | (df['vertical_meters'] == 0)) &
    (df['stage_nr'] == 20)
]
unique_missing_20 = missing_20[['year', 'race', 'distance', 'url']].drop_duplicates()

print(unique_missing_20.to_string(index=False))


 year           race  distance                                                               url
 2006 tour-de-france     154.0 https://www.procyclingstats.com/race/tour-de-france/2006/stage-20
 2007 tour-de-france     146.0 https://www.procyclingstats.com/race/tour-de-france/2007/stage-20
 2010 tour-de-france     102.0 https://www.procyclingstats.com/race/tour-de-france/2010/stage-20
 2005  giro-d-italia     119.0  https://www.procyclingstats.com/race/giro-d-italia/2005/stage-20
 2006  giro-d-italia     140.0  https://www.procyclingstats.com/race/giro-d-italia/2006/stage-20


Sind jedoch alles Schlussetappen und somit nehmen wir den Median von den Schlussetappen 21

In [42]:
flach_medians = {
    'giro-d-italia': 692,
    'tour-de-france': 732
}


def impute_specific_stage_20(row):

    if row['stage_nr'] == 20 and (pd.isna(row['vertical_meters']) or row['vertical_meters'] == 0):
        return flach_medians.get(row['race'], 725)
    return row['vertical_meters']


df['vertical_meters'] = df.apply(impute_specific_stage_20, axis=1)


df['vertical_meters'] = df['vertical_meters'].astype('Int64')


print(f"Verbleibende NaNs/0er auf Etappe 20: {len(df[(df['stage_nr'] == 20) & ((df['vertical_meters'].isna()) | (df['vertical_meters'] == 0))])}")
print(f"Verbleibende NaNs im GESAMTEN DF:    {df['vertical_meters'].isna().sum()}")

Verbleibende NaNs/0er auf Etappe 20: 0
Verbleibende NaNs im GESAMTEN DF:    0


### Prüfung 0 oder NA Werte in distance oder vertical_meters

In [ ]:


distance_nans = df['distance'].isna().sum()
distance_zeros = (df['distance'] == 0).sum()


vm_nans = df['vertical_meters'].isna().sum()
vm_zeros = (df['vertical_meters'] == 0).sum()

print(f"Spalte 'distance':")
print(f"  - Fehlende Werte (NaN): {distance_nans}")
print(f"  - Null-Werte (0):       {distance_zeros}\n")

print(f"Spalte 'vertical_meters':")
print(f"  - Fehlende Werte (NaN): {vm_nans}")
print(f"  - Null-Weters (0):      {vm_zeros}")

Spalte 'distance':
  - Fehlende Werte (NaN): 0
  - Null-Werte (0):       0

Spalte 'vertical_meters':
  - Fehlende Werte (NaN): 0
  - Null-Weters (0):      0


In [50]:
print(df.isna().sum())

# Hilfsspalten entfernen
spalten_zum_loeschen = ['distance_recherchiert_x', 'distance_recherchiert_y']
df.drop(columns=spalten_zum_loeschen, inplace=True, errors='ignore')

race                                0
year                                0
url                                 0
rank                                0
rider_url                           0
time_gap                            0
birthdate                           0
height                            910
name                                0
nationality                         0
weight                            910
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                     0
won_how                             0
avg_speed                           0
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr                            0
date        

### Checkpoint
Neues .pkl erstellen 13

In [48]:
pfad = '../../data/processed/13_cleaned_master_data.pkl'
df.to_pickle(pfad)